# Rate constant Calculation and Diagrams

In [1]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import sys
sys.path.append("/Users/tdinelli/Documents/GitHub/diffPLOG2TROE")
from diffPLOG2TROE.rate_constants.arrhenius import Arrhenius
from diffPLOG2TROE.rate_constants.falloff import FallOff
from diffPLOG2TROE.rate_utils.rate_interpreter import parse_rate_constant

## Modified Arrhenius
This tutorial demonstrates the computation of kinetic rate constants for a simple Arrhenius reaction across a specified temperature range. The implementation focuses solely on the forward reaction rate constant, as determined by user-provided parameters. Note that this code does not incorporate chemical equilibrium calculations, thus considering only unidirectional reaction kinetics.
- Equation
    > $k_f(T) = A \: T^b \: exp\left(\frac{Eact}{RT}\right)$
- **CHEMKIN** representation
    >```
    >H2 + O = H + OH    +5.080E+04 +2.670E+00 +6.292E+03
    >```
- Internal representation
    >```python
    >rate_constant = {
    >    "name": "H2+O=H+OH",
    >    "type": "arrhenius",
    >    "rate-constant": {
    >        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    >    }
    >}
    >```
    >As an alternative option, the "name" of the reaction can be specified as follows to facilitate subsequent handling during the plotting operations.
    >```python
    >rate_constant = {
    >    "name": "\\ce{H2 + O = H + OH}",
    >    "type": "arrhenius",
    >    "rate-constant": {
    >        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    >    }
    >}
    >```

In [2]:
rate_constant = {
    "name": "H2+O=H+OH",
    "type": "arrhenius",
    "rate-constant": {
        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    }
}

constant = Arrhenius(rate_constant)
print(constant.kinetic_constant(jnp.array([500, 600, 1000])))
Arrhenius.lnA = 5

[1.45095773e+09 6.78388533e+09 2.19095915e+11]


## Fall-Off Reactions
- Equations
    > $k_f \: (T, P_r) = k_{\infty} \left(\dfrac{P_{r}}{1+P_{r}}\right)F(T, P_r) $
    >
    > $P_r$ is the reduce pressure computed as follow:
    > $P_r = \dfrac{k_{0}[M]}{k_{\infty}}$
    > 
    > $[M]$ is the concentration of the mixture, possibly including enhanced third-body efficiencies.
    >
    > $k_0$ and $k_{\infty}$ are the low pressure and high pressure limits of the rate constant computed as a classic Arrhenius constant (see above).
    >
    > $F(T, P_r)$ is the falloff function that depending on different formalism can assume different values:
    > | | |
    > |:- |:- |
    > | **Lindemann** | $F = 1$|
    > | **Troe**      | $log_{10}F = \dfrac{log_{10}Fcent}{1 + f_{1}^2}$<br/><br/>$Fcent = (1-A) exp\left(-\dfrac{T}{T_3}\right) + Aexp\left(-\dfrac{T}{T_1}\right) + exp\left(\dfrac{T_2}{T}\right)$<br/><br/>$f_1 = \dfrac{log_{10}P_{r} + c}{n - 0.14\left(log_{10}P_{r} + c\right)}$<br/><br/>$c = -0.4 - 0.67log_{10}Fcent$<br/><br/>$n = 0.75 - 1.27 log_{10}Fcent$ |
    > | **SRI**       | $X = \dfrac{1}{1 + log_{10}^{2}P_{r}}$<br/><br/>$F = d\left[a\times exp\left(-b/T\right) + exp\left(-T/c\right)\right]^{X} T^{e}$ |


- **CHEMKIN** representation (Examples are courtesy of the CHEMKIN manual)
    - **Lindemann**
      >```
      > H + C2H4(+M) = C2H5(+M)    0.221E+14  0.000  2066.0   ! Michael
      >  LOW /                     6.369E+27 -2.760 -54.000 / ! Lindemann fall-off reaction
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                         ! enhanced third-body efficiencies
      >```
    - **TROE**
      >```
      > CH3 + CH3(+M) = C2H6(+M)   9.030E+16 -1.180 654.000 
      >  LOW /                     3.180E+41 -7.030 2762.00 /
      > TROE / 0.6041 6927.00 132.00 0.000                  / ! TROE fall-off reaction, with 4 parameters the fourth one is optional
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                         ! enhanced third-body efficiencies
      >```
    - **SRI**
      >```
      > CH3 + H(+M) = CH4(+M)      6.000E+16 -1.000 0.000 
      >  LOW /                     8.000E+26 -3.000 0.000 /
      > SRI  / 0.450 797.00 979.00 0.000 0.000            / ! SRI fall-off reaction
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                       ! enhanced third-body efficiencies
      >```

- Internal representation
    >```python
    >lindemann_constant = {
    >    "name": "H+C2H4(+M)=C2H5(+M)",
    >    "type": "falloff",
    >    "falloff-type": "lindemann",
    >    "rate-constant": {
    >        "lpl-coefficients": [6.369e+27, -2.760, -54.000],
    >        "hpl-coefficients": [0.221e+14, 0.000, 2066.0],
    >    }
    >}
    >
    >troe_constant = {
    >    "name": "CH3+CH3(+M)=C2H6(+M)",
    >    "type": "falloff",
    >    "falloff-type": "troe",
    >    "rate-constant": {
    >        "lpl-coefficients": [9.030e+16, -1.180, 654.000],
    >        "hpl-coefficients": [6.369e+27, -2.760, -54.000],
    >        "falloff-coefficients": [0.6041, 6927.00, 132.00],
    >        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5
    >    }
    >}
    >
    >sri_constant = {
    >    "name": "CH3+H(+M)=CH4(+M)",
    >    "type": "falloff",
    >    "falloff-type": "sri",
    >    "rate-constant": {
    >        "lpl-coefficients": [6.000e+16, -1.000, 0.000],
    >        "hpl-coefficients": [8.000e+26, -3.000, 0.000],
    >        "falloff-coefficients": [0.450, 797.00, 979.00],
    >        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    >    }
    >}
```

In [3]:
lindemann_constant = {
    "name": "H+C2H4(+M)=C2H5(+M)",
    "type": "falloff",
    "falloff-type": "lindemann",
    "rate-constant": {
        "lpl-coefficients": [6.369e+27, -2.760, -54.000],
        "hpl-coefficients": [0.221e+14, 0.000, 2066.0],
    }
}

troe_constant = {
    "name": "CH3+CH3(+M)=C2H6(+M)",
    "type": "falloff",
    "falloff-type": "troe",
    "rate-constant": {
        "lpl-coefficients": [9.030E+16, -1.180, 654.000],
        "hpl-coefficients": [6.369E+27, -2.760, -54.000],
        "falloff-coefficients": [0.6041, 6927.00, 132.00, 0.0],
        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    }
}

constant = FallOff(troe_constant)
print(constant.kinetic_constant(jnp.array([500, 600, 1000]), 1.))

sri_constant = {
    "name": "CH3+H(+M)=CH4(+M)",
    "type": "falloff",
    "falloff-type": "sri",
    "rate-constant": {
        "lpl-coefficients": [6.000e+16, -1.000, 0.000],
        "hpl-coefficients": [8.000e+26, -3.000, 0.000],
        "falloff-coefficients": [0.450, 797.00, 979.00],
        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    }
}

[7.03560714e+08 5.25987415e+08 2.13504590e+08]
